In [35]:
import pandas as pd
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from utils.preprocess import create_features
from utils.data_split import split_data

In [36]:
df = pd.read_csv('../../../data/creditcard.csv')
df = create_features(df)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0,5.014760,0,1,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,0,1.305626,0,1,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,5.939276,0,1,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0,4.824306,0,1,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,0,4.262539,0,1,0


In [37]:
features = [c for c in df.columns if c.startswith("V")] + [
    'Amount','_log_amount',
    'Hour_from_start_mod24','is_night_proxy','is_business_hours_proxy'
]
target = "Class"

In [38]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, target)

X_train:               V1        V2        V3        V4        V5        V6        V7  \
0      -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1       1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2      -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3      -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4      -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   
...          ...       ...       ...       ...       ...       ...       ...   
181579  1.854479 -1.811512  0.701881  0.020085 -2.102147  0.975199 -2.048749   
181580  1.736338 -0.538109 -1.813615  0.521401  0.023036 -1.317617  0.711378   
181581 -1.888196  0.614259  0.363199  0.656021  1.261909  1.034714 -0.247814   
181582  1.957013 -0.522989 -0.390832  0.311711 -0.597066 -0.046566 -0.736245   
181583 -0.625854  0.582609  1.561304  4.640512  0.472191  0.591272  0.357187   

              V8        V9    

In [39]:
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
lgbm_pipe = ImbPipeline(steps=[
    ("model", LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        scale_pos_weight=neg/max(pos, 1),
    ))
])

lgbm_pipe.fit(X_train, y_train)

val_lgb_proba  = lgbm_pipe.predict_proba(X_val)[:,1]
test_lgb_proba = lgbm_pipe.predict_proba(X_test)[:,1]

NameError: name 'SEED' is not defined